In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import faiss    
from sentence_transformers import SentenceTransformer

def find_ai_lab_root(start: Path) -> Path:
    current = start.resolve()
    for candidate in [current] + list(current.parents):
        if candidate.name == "ai_lab":
            return candidate
    raise RuntimeError("Không tìm thấy thư mục ai_lab.")

AI_LAB_ROOT = find_ai_lab_root(Path.cwd())

ARTIFACTS_DIR = AI_LAB_ROOT / "artifacts" / "retriever_v1"
DATASETS_DIR = AI_LAB_ROOT / "datasets"
REPORTS_DIR = AI_LAB_ROOT / "reports"

KB_CHUNKS_PATH = ARTIFACTS_DIR / "kb_chunks_v1.json"
FAISS_INDEX_PATH = ARTIFACTS_DIR / "faiss.index"
EMBEDDING_CONFIG_PATH = ARTIFACTS_DIR / "embedding_config.json"
EVAL_SET_PATH = DATASETS_DIR / "eval" / "health_rag_eval_v1_1.json"
EVAL_REPORT_PATH = REPORTS_DIR / "retrieval_eval_v1_1.csv"

print("AI_LAB_ROOT =", AI_LAB_ROOT)

AI_LAB_ROOT = D:\homelab\ai_lab


In [9]:
with open(KB_CHUNKS_PATH, "r", encoding="utf-8") as f:
    kb_chunks = json.load(f)

with open(EMBEDDING_CONFIG_PATH, "r", encoding="utf-8") as f:
    embedding_config = json.load(f)

with open(EVAL_SET_PATH, "r", encoding="utf-8") as f:
    eval_set = json.load(f)

index = faiss.read_index(str(FAISS_INDEX_PATH))
model = SentenceTransformer(embedding_config["model_name"])

print("Chunks:", len(kb_chunks))
print("Eval queries:", len(eval_set))
print("Index total:", index.ntotal)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: intfloat/multilingual-e5-small
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Chunks: 15
Eval queries: 40
Index total: 15


In [10]:
def search(query, top_k=3):
    query_text = embedding_config["query_prefix"] + query.strip()
    q_emb = model.encode(
        [query_text],
        convert_to_numpy=True,
        normalize_embeddings=True
    ).astype("float32")

    scores, indices = index.search(q_emb, top_k)

    results = []
    for score, idx in zip(scores[0], indices[0]):
        if idx == -1:
            continue
        chunk = kb_chunks[idx]
        results.append({
            "score": float(score),
            "chunk_id": chunk["chunk_id"],
            "source_id": chunk["source_id"],
            "section": chunk["section"],
            "title": chunk["title"]
        })
    return results

In [11]:
rows = []

for sample in eval_set:
    query = sample["query"]
    expected_chunk_ids = set(sample["expected_chunk_ids"])
    expected_source_id = sample["expected_source_id"]
    expected_section = sample["expected_section"]

    results = search(query, top_k=3)

    top1 = results[0] if len(results) > 0 else None
    top3_chunk_ids = [r["chunk_id"] for r in results[:3]]

    hit_at_1 = int(top1 is not None and top1["chunk_id"] in expected_chunk_ids)
    hit_at_3 = int(any(cid in expected_chunk_ids for cid in top3_chunk_ids))
    source_at_1 = int(top1 is not None and top1["source_id"] == expected_source_id)
    section_at_1 = int(top1 is not None and top1["section"] == expected_section)

    rows.append({
        "query": query,
        "expected_chunk_ids": ",".join(expected_chunk_ids),
        "expected_source_id": expected_source_id,
        "expected_section": expected_section,
        "top1_chunk_id": top1["chunk_id"] if top1 else None,
        "top1_source_id": top1["source_id"] if top1 else None,
        "top1_section": top1["section"] if top1 else None,
        "top1_score": top1["score"] if top1 else None,
        "top3_chunk_ids": ",".join(top3_chunk_ids),
        "hit_at_1": hit_at_1,
        "hit_at_3": hit_at_3,
        "source_at_1": source_at_1,
        "section_at_1": section_at_1
    })

df_eval = pd.DataFrame(rows)
df_eval.to_csv(EVAL_REPORT_PATH, index=False, encoding="utf-8-sig")

print("Đã ghi:", EVAL_REPORT_PATH)
df_eval.head()

Đã ghi: D:\homelab\ai_lab\reports\retrieval_eval_v1_1.csv


,query,expected_chunk_ids,expected_source_id,expected_section,top1_chunk_id,top1_source_id,top1_section,top1_score,top3_chunk_ids,hit_at_1,hit_at_3,source_at_1,section_at_1
0,xét nghiệm máu là gì,kb_v1_005_c1,blood_tests,test_explainers,kb_v1_005_c1,blood_tests,test_explainers,0.939669,"kb_v1_005_c1,kb_v1_006_c1,kb_v1_007_c1",1,1,1,1
1,blood test là gì,kb_v1_005_c1,blood_tests,test_explainers,kb_v1_005_c1,blood_tests,test_explainers,0.906717,"kb_v1_005_c1,kb_v1_006_c1,kb_v1_007_c1",1,1,1,1
2,vì sao bác sĩ chỉ định xét nghiệm máu,kb_v1_006_c1,blood_tests,test_explainers,kb_v1_006_c1,blood_tests,test_explainers,0.938549,"kb_v1_006_c1,kb_v1_008_c1,kb_v1_007_c1",1,1,1,1
3,lý do cần làm xét nghiệm máu,kb_v1_006_c1,blood_tests,test_explainers,kb_v1_006_c1,blood_tests,test_explainers,0.937595,"kb_v1_006_c1,kb_v1_005_c1,kb_v1_008_c1",1,1,1,1
4,một số loại xét nghiệm máu thường gặp,kb_v1_007_c1,blood_tests,test_explainers,kb_v1_007_c1,blood_tests,test_explainers,0.937207,"kb_v1_007_c1,kb_v1_005_c1,kb_v1_009_c1",1,1,1,1


In [12]:
metrics = {
    "Recall@1": df_eval["hit_at_1"].mean(),
    "Recall@3": df_eval["hit_at_3"].mean(),
    "Source Accuracy@1": df_eval["source_at_1"].mean(),
    "Section Accuracy@1": df_eval["section_at_1"].mean(),
}

pd.DataFrame([metrics])

,Recall@1,Recall@3,Source Accuracy@1,Section Accuracy@1
0,0.95,1.0,0.975,1.0


In [13]:
wrong_top1 = df_eval[df_eval["hit_at_1"] == 0].copy()
wrong_top1

,query,expected_chunk_ids,expected_source_id,expected_section,top1_chunk_id,top1_source_id,top1_section,top1_score,top3_chunk_ids,hit_at_1,hit_at_3,source_at_1,section_at_1
34,"người bệnh nhiễm trùng xấu đi nhanh, tím môi v...","kb_v1_003_c1,kb_v1_001_c1,kb_v1_013_c1",nice_sepsis_overview,red_flags,kb_v1_002_c1,nice_sepsis_overview,red_flags,0.886667,"kb_v1_002_c1,kb_v1_003_c1,kb_v1_001_c1",0,1,1,1
35,"da tái xám, ban không mất màu và rất mệt có ng...","kb_v1_001_c1,kb_v1_002_c1",nice_sepsis_overview,red_flags,kb_v1_003_c1,nice_sepsis_overview,red_flags,0.896224,"kb_v1_003_c1,kb_v1_002_c1,kb_v1_001_c1",0,1,1,1


In [14]:
wrong_top3 = df_eval[df_eval["hit_at_3"] == 0].copy()
wrong_top3

,query,expected_chunk_ids,expected_source_id,expected_section,top1_chunk_id,top1_source_id,top1_section,top1_score,top3_chunk_ids,hit_at_1,hit_at_3,source_at_1,section_at_1
